El script debe realizar un agrupamiento o clustering con K-means enfocado en características sociodemográficas
Consignas:

    1. Seleccionar las variables numéricas del dataset para clustering.

    2. Escalar si es necesario (normalizar)

    3. Aplicar K-means con distintos valores de k.

    4. Determinar el número óptimo de clusters usando **Coeficiente de Silhouette**

    5. **ANÁLISIS SOCIODEMOGRÁFICO POST-CLUSTERING**: Analizar las características sociodemográficas (income, age, sex, native-country) de cada cluster para identificar patrones de ingresos

**(fnlwgt usado solo para ordenar - es peso de muestreo, no característica)**

### Variables numéricas:
 - age 
 - education-num 
 - **neto** (capital-gain - capital-loss) ← Columna combinada para reducir ceros
 - hours-per-week

### Preprocesamiento:
- Se combinan `capital-gain` y `capital-loss` en una sola columna **neto** = gain - loss
- Los datos se ordenan por `fnlwgt` (mayor a menor) para priorizar registros con mayor peso muestral
- `fnlwgt` NO se usa en el análisis de clustering

## Paso 1: Carga de datos, preprocesamiento y selección de variables numéricas


In [1]:
import sys
sys.path.append('..')
from utils import cargar_dataset_adult
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Configuración de visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

# Cargar dataset
df = cargar_dataset_adult()

print("="*60)
print("PREPROCESAMIENTO Y SELECCIÓN DE VARIABLES")
print("="*60)

# PASO 1: Crear columna "neto" combinando capital-gain y capital-loss
# Esto reduce la cantidad de ceros y crea una variable más informativa
df['neto'] = df['capital-gain'] - df['capital-loss']

print("\n1. Creación de columna 'neto' (capital-gain - capital-loss):")
print(f"   • Valores originales con ceros:")
print(f"     - capital-gain: {(df['capital-gain'] == 0).sum()} ceros de {len(df)} ({(df['capital-gain'] == 0).sum()/len(df)*100:.1f}%)")
print(f"     - capital-loss: {(df['capital-loss'] == 0).sum()} ceros de {len(df)} ({(df['capital-loss'] == 0).sum()/len(df)*100:.1f}%)")
print(f"   • Columna 'neto': {(df['neto'] == 0).sum()} ceros de {len(df)} ({(df['neto'] == 0).sum()/len(df)*100:.1f}%)")
print(f"   ✓ Ventaja: Representa ganancia/pérdida neta de capital en una sola variable")

# PASO 2: Ordenar por fnlwgt (mayor a menor)
# fnlwgt es el peso de muestreo del censo; valores altos = más representativos
df = df.sort_values('fnlwgt', ascending=False).reset_index(drop=True)

print(f"\n2. Ordenamiento por 'fnlwgt' (de mayor a menor):")
print(f"   • Rango fnlwgt: [{df['fnlwgt'].min():.0f}, {df['fnlwgt'].max():.0f}]")
print(f"   • Primeros registros tienen mayor peso muestral (más representativos)")
print(f"   ✓ Nota: fnlwgt NO se usa para el clustering, solo para ordenar")

# PASO 3: Seleccionar variables numéricas para el análisis
numeric_cols = ['age', 'education-num', 'neto', 'hours-per-week']

# Extraer variables numéricas
X = df[numeric_cols].copy()

print(f"\n3. Variables numéricas seleccionadas para K-means:")
print(f"   {numeric_cols}")
print(f"\n{'='*60}")
print(f"DATASET FINAL PARA CLUSTERING")
print(f"{'='*60}")
print(f"\nDimensión: {X.shape}")
print(f"\nEstadísticas descriptivas:\n")
print(X.describe())


PREPROCESAMIENTO Y SELECCIÓN DE VARIABLES

1. Creación de columna 'neto' (capital-gain - capital-loss):
   • Valores originales con ceros:
     - capital-gain: 44807 ceros de 48842 (91.7%)
     - capital-loss: 46560 ceros de 48842 (95.3%)
   • Columna 'neto': 42525 ceros de 48842 (87.1%)
   ✓ Ventaja: Representa ganancia/pérdida neta de capital en una sola variable

2. Ordenamiento por 'fnlwgt' (de mayor a menor):
   • Rango fnlwgt: [12285, 1490400]
   • Primeros registros tienen mayor peso muestral (más representativos)
   ✓ Nota: fnlwgt NO se usa para el clustering, solo para ordenar

3. Variables numéricas seleccionadas para K-means:
   ['age', 'education-num', 'neto', 'hours-per-week']

DATASET FINAL PARA CLUSTERING

Dimensión: (48842, 4)

Estadísticas descriptivas:

              age  education-num        neto  hours-per-week
count  48842.0000     48842.0000  48842.0000      48842.0000
mean      38.6436        10.0781    991.5653         40.4224
std       13.7105         2.5710   

## Paso 2: Normalización/Escalado

K-means es sensible a la escala de las variables, por lo que es necesario normalizar los datos usando **StandardScaler** (Z-score normalization).


In [2]:
# Normalizar usando StandardScaler (Z-score)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convertir a DataFrame para mejor visualización
X_scaled_df = pd.DataFrame(X_scaled, columns=numeric_cols, index=X.index)

print("="*60)
print("NORMALIZACIÓN DE DATOS (StandardScaler)")
print("="*60)
print("\nEstadísticas ANTES de normalizar:")
print(X.describe().loc[['mean', 'std']])

print("\n\nEstadísticas DESPUÉS de normalizar:")
# NOTA: StandardScaler garantiza matemáticamente que media=0 y std=1.
# Sin embargo, debido a errores de precisión de punto flotante (float64),
# la media puede aparecer como valores muy pequeños (ej: 2.28e-16) en lugar de 0.
# Estos valores son efectivamente cero (16 órdenes de magnitud menores que los datos originales).
# 
# Usamos np.round() con 10 decimales para:
# 1. Mostrar 0.0 en lugar de notación científica (mejora legibilidad)
# 2. No afectar los cálculos posteriores (el redondeo a 10 decimales es despreciable)
# 3. Confirmar visualmente que la normalización funcionó correctamente
stats_after = X_scaled_df.describe().loc[['mean', 'std']]
stats_after_rounded = np.round(stats_after, 10)
print(stats_after_rounded)




NORMALIZACIÓN DE DATOS (StandardScaler)

Estadísticas ANTES de normalizar:
          age  education-num       neto  hours-per-week
mean  38.6436        10.0781   991.5653         40.4224
std   13.7105         2.5710  7475.5499         12.3914


Estadísticas DESPUÉS de normalizar:
      age  education-num  neto  hours-per-week
mean  0.0           -0.0  -0.0             0.0
std   1.0            1.0   1.0             1.0


## Paso 3: Aplicar K-means con distintos valores de k

Probamos K-means con k desde 2 hasta 10 clusters.


In [3]:
# Rango de k a probar
k_range = range(2, 11)

# Listas para almacenar resultados
silhouette_scores = []
kmeans_models = {}

print("="*60)
print("APLICANDO K-MEANS CON DISTINTOS VALORES DE K")
print("="*60)
print("\nCalculando Coeficiente de Silhouette...\n")
print(" Silhouette mide la calidad del clustering considerando:")
print("   • Cohesión: qué tan cerca están los puntos de su propio cluster")
print("   • Separación: qué tan lejos están de otros clusters")
print("   • Valores: [-1, 1] → Maximizar (cercano a 1 es mejor)\n")

# Aplicar K-means para cada valor de k
for k in k_range:
    # Entrenar K-means
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    
    # Guardar el modelo
    kmeans_models[k] = kmeans
    
    # Calcular Coeficiente de Silhouette
    silhouette = silhouette_score(X_scaled, labels)
    silhouette_scores.append(silhouette)
    
    print(f"k={k:2d} | Silhouette: {silhouette:.4f}")

print("\n✓ Entrenamiento completado para todos los valores de k")


APLICANDO K-MEANS CON DISTINTOS VALORES DE K

Calculando Coeficiente de Silhouette...

 Silhouette mide la calidad del clustering considerando:
   • Cohesión: qué tan cerca están los puntos de su propio cluster
   • Separación: qué tan lejos están de otros clusters
   • Valores: [-1, 1] → Maximizar (cercano a 1 es mejor)

k= 2 | Silhouette: 0.8353
k= 3 | Silhouette: 0.2427
k= 4 | Silhouette: 0.2857
k= 5 | Silhouette: 0.3046
k= 6 | Silhouette: 0.3100
k= 7 | Silhouette: 0.3204
k= 8 | Silhouette: 0.3214
k= 9 | Silhouette: 0.3210
k=10 | Silhouette: 0.2872

✓ Entrenamiento completado para todos los valores de k


## Paso 4: Determinar el número óptimo de clusters

Usamos el **Coeficiente de Silhouette** para determinar el k óptimo:

- **Objetivo**: Maximizar (valores cercanos a 1 son mejores)
- **Significado**: Mide la calidad del clustering balanceando cohesión interna y separación externa
- **Ventaja**: Proporciona un valor óptimo claro y es intuitivo de interpretar


In [4]:
# Determinar k óptimo según Silhouette (maximizar)
k_optimal = k_range[np.argmax(silhouette_scores)]
max_silhouette = max(silhouette_scores)

# Resumen de resultados
print("="*60)
print("DETERMINACIÓN DEL NÚMERO ÓPTIMO DE CLUSTERS")
print("="*60)

# Tabla resumen
results_df = pd.DataFrame({
    'k': list(k_range),
    'Silhouette': silhouette_scores
})

print(f"\nTabla Resumen de Coeficiente de Silhouette:")
print(results_df.to_string(index=False))

# Marcar el óptimo
print(f"\n{'='*60}")
print(f"✓ K ÓPTIMO = {k_optimal}")
print(f"{'='*60}")
print(f"\nCoeficiente de Silhouette: {max_silhouette:.4f}")
print(f"\nInterpretación:")
if max_silhouette > 0.7:
    print("   → Estructura de clustering FUERTE")
elif max_silhouette > 0.5:
    print("   → Estructura de clustering RAZONABLE")
elif max_silhouette > 0.25:
    print("   → Estructura de clustering DÉBIL pero presente")
else:
    print("   → Estructura de clustering ARTIFICIAL (no hay clusters naturales claros)")


DETERMINACIÓN DEL NÚMERO ÓPTIMO DE CLUSTERS

Tabla Resumen de Coeficiente de Silhouette:
 k  Silhouette
 2      0.8353
 3      0.2427
 4      0.2857
 5      0.3046
 6      0.3100
 7      0.3204
 8      0.3214
 9      0.3210
10      0.2872

✓ K ÓPTIMO = 2

Coeficiente de Silhouette: 0.8353

Interpretación:
   → Estructura de clustering FUERTE


## Paso 5: Análisis del modelo óptimo

Analizamos los clusters usando el k óptimo determinado por las métricas.


In [5]:
# Usar el k óptimo según Silhouette
kmeans_optimal = kmeans_models[k_optimal]
labels_optimal = kmeans_optimal.labels_

# Agregar labels al DataFrame original
df_clustered = df[numeric_cols].copy()
df_clustered['Cluster'] = labels_optimal

print("="*60)
print(f"ANÁLISIS DEL MODELO ÓPTIMO (k={k_optimal})")
print("="*60)

# Estadísticas por cluster
print(f"\n Distribución de observaciones por cluster:")
cluster_counts = pd.Series(labels_optimal).value_counts().sort_index()
for cluster_id, count in cluster_counts.items():
    percentage = (count / len(labels_optimal)) * 100
    print(f"   Cluster {cluster_id}: {count:6d} observaciones ({percentage:5.2f}%)")

# Características promedio por cluster
print(f"\n\n Características promedio por cluster:\n")
cluster_summary = df_clustered.groupby('Cluster')[numeric_cols].mean()
print(cluster_summary.round(2))

print(f"\n\n✓ Análisis completado con k óptimo = {k_optimal}")


ANÁLISIS DEL MODELO ÓPTIMO (k=2)

 Distribución de observaciones por cluster:
   Cluster 0:  48598 observaciones (99.50%)
   Cluster 1:    244 observaciones ( 0.50%)


 Características promedio por cluster:

           age  education-num      neto  hours-per-week
Cluster                                                
0        38.60          10.06    494.47           40.37
1        46.61          13.02  99999.00           50.48


✓ Análisis completado con k óptimo = 2


## Paso 6: Análisis Sociodemográfico Post-Clustering

**Objetivo:** Analizar las características sociodemográficas de cada cluster para identificar patrones de ingresos y perfiles sociodemográficos.


In [6]:
# Análisis sociodemográfico post-clustering
print("="*70)
print("ANÁLISIS SOCIODEMOGRÁFICO POST-CLUSTERING")
print("="*70)

# Obtener el modelo óptimo y las etiquetas
kmeans_optimal = kmeans_models[k_optimal]
cluster_labels = kmeans_optimal.labels_

# Agregar las etiquetas de cluster al dataframe original
df['cluster'] = cluster_labels

print(f"\nClusters identificados: {k_optimal}")
print(f"Distribución de observaciones por cluster:")
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
print(cluster_counts)

# 1. ANÁLISIS DE INGRESOS POR CLUSTER
print("\n" + "="*50)
print("1. ANÁLISIS DE INGRESOS POR CLUSTER")
print("="*50)

# Tabla de contingencia: cluster vs income
income_by_cluster = pd.crosstab(df['cluster'], df['income'], margins=True)
print("\nDistribución de ingresos por cluster:")
print(income_by_cluster)

# Porcentajes por cluster
income_pct_by_cluster = pd.crosstab(df['cluster'], df['income'], normalize='index') * 100
print("\nPorcentajes de ingresos por cluster:")
print(income_pct_by_cluster.round(2))

# Identificar cluster con mayor proporción de ingresos altos
high_income_clusters = income_pct_by_cluster['>50K'].sort_values(ascending=False)
print(f"\nClusters ordenados por % de ingresos altos:")
for cluster, pct in high_income_clusters.items():
    print(f"Cluster {cluster}: {pct:.1f}% ingresos altos")

# 2. ANÁLISIS DE EDAD POR CLUSTER
print("\n" + "="*50)
print("2. ANÁLISIS DE EDAD POR CLUSTER")
print("="*50)

age_by_cluster = df.groupby('cluster')['age'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
print("\nEstadísticas de edad por cluster:")
print(age_by_cluster.round(2))

# Crear grupos de edad para análisis más detallado
df['age_group'] = pd.cut(df['age'], bins=[0, 30, 40, 50, 60, 100], 
                        labels=['<30', '30-39', '40-49', '50-59', '60+'])

age_group_by_cluster = pd.crosstab(df['cluster'], df['age_group'], normalize='index') * 100
print("\nDistribución de grupos de edad por cluster (%):")
print(age_group_by_cluster.round(1))

# 3. ANÁLISIS DE GÉNERO POR CLUSTER
print("\n" + "="*50)
print("3. ANÁLISIS DE GÉNERO POR CLUSTER")
print("="*50)

gender_by_cluster = pd.crosstab(df['cluster'], df['sex'], normalize='index') * 100
print("\nDistribución de género por cluster (%):")
print(gender_by_cluster.round(1))


# 5. PERFILES SOCIODEMOGRÁFICOS DE CLUSTERS
print("\n" + "="*50)
print("5. PERFILES SOCIODEMOGRÁFICOS DE CLUSTERS")
print("="*50)

for cluster in range(k_optimal):
    cluster_data = df[df['cluster'] == cluster]
    print(f"\n--- CLUSTER {cluster} ---")
    print(f"Tamaño: {len(cluster_data)} observaciones ({len(cluster_data)/len(df)*100:.1f}%)")
    
    # Ingresos
    high_income_pct = (cluster_data['income'] == '>50K').mean() * 100
    print(f"Ingresos altos: {high_income_pct:.1f}%")
    
    # Edad
    avg_age = cluster_data['age'].mean()
    print(f"Edad promedio: {avg_age:.1f} años")
    
    # Género
    male_pct = (cluster_data['sex'] == 'Male').mean() * 100
    print(f"Género masculino: {male_pct:.1f}%")
    

# 6. CONCLUSIONES SOCIODEMOGRÁFICAS
print("\n" + "="*50)
print("6. CONCLUSIONES SOCIODEMOGRÁFICAS")
print("="*50)

print("\nPatrones identificados:")
print("-" * 30)

# Encontrar el cluster con mayor proporción de ingresos altos
best_income_cluster = high_income_clusters.index[0]
best_income_pct = high_income_clusters.iloc[0]

print(f"\n1. CLUSTER CON MAYOR INGRESO: Cluster {best_income_cluster}")
print(f"   - {best_income_pct:.1f}% de personas con ingresos >$50K")
print(f"   - Edad promedio: {df[df['cluster']==best_income_cluster]['age'].mean():.1f} años")
print(f"   - Género: {(df[df['cluster']==best_income_cluster]['sex']=='Male').mean()*100:.1f}% masculino")

# Análisis de diferencias entre clusters
print(f"\n2. DIFERENCIAS PRINCIPALES ENTRE CLUSTERS:")
for i in range(k_optimal):
    for j in range(i+1, k_optimal):
        cluster_i_data = df[df['cluster'] == i]
        cluster_j_data = df[df['cluster'] == j]
        
        income_diff = (cluster_i_data['income'] == '>50K').mean() - (cluster_j_data['income'] == '>50K').mean()
        age_diff = cluster_i_data['age'].mean() - cluster_j_data['age'].mean()
        
        print(f"   Cluster {i} vs Cluster {j}:")
        print(f"   - Diferencia en ingresos altos: {income_diff*100:+.1f}%")
        print(f"   - Diferencia en edad promedio: {age_diff:+.1f} años")

print(f"\n" + "="*70)
print("FIN DEL ANÁLISIS SOCIODEMOGRÁFICO")
print("="*70)


ANÁLISIS SOCIODEMOGRÁFICO POST-CLUSTERING

Clusters identificados: 2
Distribución de observaciones por cluster:
0    48598
1      244
Name: count, dtype: int64

1. ANÁLISIS DE INGRESOS POR CLUSTER

Distribución de ingresos por cluster:
income   <=50K   >50K    All
cluster                     
0        37155  11443  48598
1            0    244    244
All      37155  11687  48842

Porcentajes de ingresos por cluster:
income   <=50K    >50K
cluster               
0        76.45   23.55
1         0.00  100.00

Clusters ordenados por % de ingresos altos:
Cluster 1: 100.0% ingresos altos
Cluster 0: 23.5% ingresos altos

2. ANÁLISIS DE EDAD POR CLUSTER

Estadísticas de edad por cluster:
         count   mean  median    std  min  max
cluster                                       
0        48598  38.60    37.0  13.71   17   90
1          244  46.61    47.0  10.68   21   80

Distribución de grupos de edad por cluster (%):
age_group   <30  30-39  40-49  50-59  60+
cluster                         